#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze table

In [0]:
admissions_2021 = spark.table("health.bronze.admissions_2021")
admissions_2022 = spark.table("health.bronze.admissions_2022")
admissions_2023 = spark.table("health.bronze.admissions_2023")
admissions_2024 = spark.table("health.bronze.admissions_2024")
admissions_2025 = spark.table("health.bronze.admissions_2025")

#Data Understanding

###Review Schema

In [0]:
admissions_2021.printSchema()

###Review Sample Records


In [0]:
display(admissions_2021.limit(10))

#Data Consolidation

In [0]:
admissions = (
    admissions_2021
    .unionByName(admissions_2022)
    .unionByName(admissions_2023)
    .unionByName(admissions_2024)
    .unionByName(admissions_2025)
    .dropDuplicates()
)

#Data Profiling

###Check Null Values

In [0]:
display(
    admissions.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in admissions.columns
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = admissions.count()

distinct_admission_ids = (
    admissions
    .select("admission_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Admission IDs: {distinct_admission_ids}")

###Data Quality Analysis

In [0]:
display(
    admissions.select(
        F.min("admission_id").alias("min_admission_id"),
        F.min("patient_id").alias("min_patient_id"),
        F.min("hospital_id").alias("min_hospital_id"),
        F.min("doctor_id").alias("min_doctor_id"),
        F.min("procedure_id").alias("min_procedure_id")
    )
)

#Transformations

###Separate Valid and Invalid Records

In [0]:
valid_admissions = admissions.filter(
    F.col("admission_date").isNotNull()
)

admissions_missing_date = admissions.filter(
    F.col("admission_date").isNull()
)

###Validation Summary

In [0]:
print(f"Valid Admissions: {valid_admissions.count()}")
print(f"Admissions Missing Date: {admissions_missing_date.count()}")
print(f"Total Admissions: {admissions.count()}")

###Review Failed Records

In [0]:
display(
    admissions_missing_date.limit(10)
)

#Write Silver Tables

###Save Valid Admissions

In [0]:
(
    valid_admissions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.admissions")
)

###Save Invalid Admissions

In [0]:
(
    admissions_missing_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.admissions_missing_date")
)

###Silver Table Verification

In [0]:
display(
    spark.table("health.silver.admissions").limit(10)
)

display(
    spark.table("health.silver.admissions_missing_date").limit(10)
)